# Mutual Fund EDA Analysis — Day 3
**Scope:** 40 schemes | 2022–2026 | Plotly + Seaborn + Matplotlib  
All charts are also exported as PNG to `reports/charts/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings, os
warnings.filterwarnings('ignore')

RAW       = '../data/raw/'
PROCESSED = '../data/processed/'
CHARTS    = '../reports/charts/'
os.makedirs(CHARTS, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PLOTLY_TEMPLATE = 'plotly_white'
print('Setup complete.')

## Load Datasets

In [ ]:
nav        = pd.read_csv(PROCESSED + '02_nav_history_cleaned.csv', parse_dates=['date'])
fund       = pd.read_csv(PROCESSED + '01_fund_master_cleaned.csv', parse_dates=['launch_date'])
aum        = pd.read_csv(PROCESSED + '03_aum_by_fund_house_cleaned.csv', parse_dates=['date'])
sip        = pd.read_csv(PROCESSED + '04_monthly_sip_inflows_cleaned.csv', parse_dates=['month'])
cat_inflow = pd.read_csv(RAW + '05_category_inflows.csv', parse_dates=['month'])
folio      = pd.read_csv(RAW + '06_industry_folio_count.csv', parse_dates=['month'])
perf       = pd.read_csv(PROCESSED + '07_scheme_performance_cleaned.csv')
txn        = pd.read_csv(PROCESSED + '08_investor_transactions_cleaned.csv', parse_dates=['transaction_date'])
holdings   = pd.read_csv(RAW + '09_portfolio_holdings.csv')
bench      = pd.read_csv(PROCESSED + '10_benchmark_indices_cleaned.csv', parse_dates=['date'] if 'date' in pd.read_csv(PROCESSED+'10_benchmark_indices_cleaned.csv', nrows=1).columns else [])

# Equity-only funds for holdings analysis
equity_codes = fund[fund['category']=='Equity']['amfi_code'].tolist()
nav_trading = nav[nav['is_trading_day']==1].copy()
print('All datasets loaded.')

---
## Finding 1: Bull Run 2023 & Correction 2024 Visible in NAV Trends
> All 40 schemes showed a sustained rally through 2023, with momentum peaking mid-2024 before a broad correction in Q3–Q4 2024 driven by FII outflows and global rate uncertainty.

In [ ]:
# Chart 1: NAV Trend for all 40 schemes (normalised to 100 at start)
nav_pivot = nav_trading.pivot_table(index='date', columns='amfi_code', values='nav')
nav_norm  = nav_pivot.div(nav_pivot.iloc[0]) * 100
x_dates   = nav_norm.index.strftime('%Y-%m-%d').tolist()

fig = go.Figure()
fund_map = fund.set_index('amfi_code')['scheme_name'].to_dict()
for col in nav_norm.columns:
    fname = fund_map.get(col, str(col))
    fig.add_trace(go.Scatter(
        x=x_dates, y=nav_norm[col].tolist(),
        mode='lines', name=str(col),
        line=dict(width=1), opacity=0.5,
        hovertemplate=f'<b>{fname[:40]}</b><br>%{{x}}<br>Idx: %{{y:.1f}}<extra></extra>'
    ))

fig.add_vrect(x0='2023-01-01', x1='2023-12-31',
    fillcolor='green', opacity=0.08, line_width=0,
    annotation_text='2023 Bull Run', annotation_position='top left',
    annotation_font=dict(color='green', size=12))

fig.add_vrect(x0='2024-09-01', x1='2024-12-31',
    fillcolor='red', opacity=0.08, line_width=0,
    annotation_text='2024 Correction', annotation_position='top left',
    annotation_font=dict(color='red', size=12))

fig.update_layout(
    title='NAV Trend - All 40 Schemes (Indexed to 100, 2022-2026)',
    xaxis_title='Date', yaxis_title='NAV Index (Base=100)',
    template=PLOTLY_TEMPLATE, height=550,
    showlegend=False, hovermode='x unified'
)
fig.write_image(CHARTS + '01_nav_trend_all_schemes.png', width=1400, height=550, scale=2)
fig.show()

---
## Finding 2: SBI Dominates Industry AUM at ₹12.5L Cr in 2025
> SBI Mutual Fund has consistently held the #1 AUM position, growing from ₹6.05L Cr in Mar 2022 to ₹12.5L Cr in Mar 2025 — nearly double its 2022 base, outpacing all peers.

In [ ]:
# Chart 2: Grouped bar chart — AUM by fund house per year
aum['year'] = aum['date'].dt.year
aum_yearly  = aum.groupby(['year','fund_house'])['aum_crore'].max().reset_index()
aum_yearly['aum_lakh_cr'] = aum_yearly['aum_crore'] / 1e5

palette = {h: ('crimson' if h=='SBI Mutual Fund' else '#5B9BD5')
           for h in aum_yearly['fund_house'].unique()}

fig, ax = plt.subplots(figsize=(16, 7))
pivot = aum_yearly.pivot(index='fund_house', columns='year', values='aum_lakh_cr').fillna(0)
pivot.plot(kind='bar', ax=ax, colormap='Blues', width=0.75, edgecolor='white')

# Re-colour SBI bars
fund_houses = pivot.index.tolist()
sbi_idx = fund_houses.index('SBI Mutual Fund')
n_years = len(pivot.columns)
for i, bar in enumerate(ax.patches):
    grp_idx = i % len(fund_houses)
    if grp_idx == sbi_idx:
        bar.set_facecolor('crimson')
        bar.set_alpha(0.9)

ax.set_title('AUM by Fund House per Year (₹ Lakh Crore) — SBI Dominance Highlighted', fontsize=14, fontweight='bold')
ax.set_xlabel('Fund House')
ax.set_ylabel('AUM (₹ Lakh Crore)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')

# Annotate SBI 2025
sbi_2025 = aum_yearly[(aum_yearly['fund_house']=='SBI Mutual Fund') & (aum_yearly['year']==2025)]
if not sbi_2025.empty:
    ax.annotate('₹12.5L Cr', xy=(sbi_idx + (n_years-1)*0.12, sbi_2025['aum_lakh_cr'].values[0]),
                xytext=(sbi_idx+1.5, sbi_2025['aum_lakh_cr'].values[0]+0.3),
                arrowprops=dict(arrowstyle='->', color='crimson'),
                fontsize=11, color='crimson', fontweight='bold')

ax.legend(title='Year', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig(CHARTS + '02_aum_grouped_bar.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Finding 3: SIP Inflows Hit All-Time High of ₹31,002 Cr in Dec 2025
> Monthly SIP inflows have grown ~2.7× from ₹11,517 Cr (Jan 2022) to ₹31,002 Cr (Dec 2025), reflecting deep retail penetration and the SIP-as-habit culture in India.

In [ ]:
# Chart 3: SIP inflow time-series with ATH annotation
sip_x = sip['month'].dt.strftime('%Y-%m-%d').tolist()
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sip_x, y=sip['sip_inflow_crore'].tolist(),
    mode='lines+markers', name='SIP Inflow',
    line=dict(color='#2196F3', width=2.5),
    marker=dict(size=4),
    fill='tozeroy', fillcolor='rgba(33,150,243,0.08)'
))

# ATH annotation
ath_idx = sip['sip_inflow_crore'].idxmax()
ath_x   = sip.loc[ath_idx, 'month'].strftime('%Y-%m-%d')
ath_y   = float(sip.loc[ath_idx, 'sip_inflow_crore'])
fig.add_annotation(
    x=ath_x, y=ath_y,
    text=f"<b>ATH Rs.{ath_y:,.0f} Cr</b><br>Dec 2025",
    showarrow=True, arrowhead=2, arrowcolor='red',
    font=dict(color='red', size=12),
    ax=40, ay=-50
)

fig.add_hline(y=20000, line_dash='dot', line_color='orange',
              annotation_text='Rs.20,000 Cr milestone', annotation_position='bottom right')

fig.update_layout(
    title='Monthly SIP Inflows (Jan 2022 - Dec 2025)',
    xaxis_title='Month', yaxis_title='SIP Inflow (Rs. Crore)',
    template=PLOTLY_TEMPLATE, height=480
)
fig.write_image(CHARTS + '03_sip_inflow_timeseries.png', width=1400, height=480, scale=2)
fig.show()

---
## Finding 4: Liquid & Sectoral/Thematic Categories Dominate Net Inflows
> The category inflow heatmap shows Liquid funds receive the highest absolute inflows every month (₹33K–₹42K Cr), while Sectoral/Thematic funds have seen a strong surge in 2024–2025.

In [ ]:
# Chart 4: Category inflow heatmap
cat_inflow['month_str'] = cat_inflow['month'].dt.strftime('%Y-%m')
heat_pivot = cat_inflow.pivot_table(index='category', columns='month_str', values='net_inflow_crore', aggfunc='sum')

fig, ax = plt.subplots(figsize=(18, 7))
sns.heatmap(heat_pivot, cmap='YlOrRd', annot=False, fmt='.0f',
            linewidths=0.3, linecolor='white', ax=ax,
            cbar_kws={'label': 'Net Inflow (₹ Crore)'})
ax.set_title('Category Inflow Heatmap — Monthly Net Inflows by Category (2024–2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Fund Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(CHARTS + '04_category_inflow_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Finding 5: 36–45 Age Group Drives the Highest SIP Volume
> The 36–45 age cohort accounts for the largest share of SIP transactions, with median ticket sizes exceeding younger cohorts — indicating wealth accumulation-phase investors are the core SIP base.

In [ ]:
# Chart 5a: Age group distribution pie chart
age_counts = txn['age_group'].value_counts().reset_index()
age_counts.columns = ['age_group','count']
age_order = ['18-25','26-35','36-45','46-55','56+']
age_counts['age_group'] = pd.Categorical(age_counts['age_group'], categories=age_order, ordered=True)
age_counts = age_counts.sort_values('age_group')

fig = px.pie(age_counts, values='count', names='age_group',
             title='Investor Age Group Distribution',
             color_discrete_sequence=px.colors.sequential.Blues_r,
             template=PLOTLY_TEMPLATE, hole=0.35)
fig.update_traces(textposition='outside', textinfo='percent+label')
fig.write_image(CHARTS + '05a_age_group_pie.png', width=700, height=500, scale=2)
fig.show()

# Chart 5b: SIP amount box plot by age group
sip_txn = txn[txn['transaction_type']=='SIP']
fig2, ax2 = plt.subplots(figsize=(10, 5))
age_order_list = ['18-25','26-35','36-45','46-55','56+']
sns.boxplot(data=sip_txn[sip_txn['age_group'].isin(age_order_list)],
            x='age_group', y='amount_inr', order=age_order_list,
            palette='Blues', showfliers=False, ax=ax2)
ax2.set_title('SIP Amount Distribution by Age Group', fontsize=13, fontweight='bold')
ax2.set_xlabel('Age Group')
ax2.set_ylabel('SIP Amount (₹)')
plt.tight_layout()
plt.savefig(CHARTS + '05b_sip_amount_boxplot_age.png', dpi=150, bbox_inches='tight')
plt.show()

# Chart 5c: Gender split
gender_counts = txn['gender'].value_counts().reset_index()
gender_counts.columns = ['gender','count']
fig3 = px.pie(gender_counts, values='count', names='gender',
              title='Investor Gender Split',
              color_discrete_map={'Male':'#2196F3','Female':'#E91E63'},
              template=PLOTLY_TEMPLATE, hole=0.4)
fig3.update_traces(textposition='outside', textinfo='percent+label')
fig3.write_image(CHARTS + '05c_gender_split_pie.png', width=600, height=450, scale=2)
fig3.show()

---
## Finding 6: Maharashtra & Delhi Lead SIP Inflows; T30 Cities Dominate
> Maharashtra and Delhi together account for over 40% of total SIP investment value, while T30 cities contribute ~75% of all transaction volume, confirming urban-led mutual fund adoption.

In [ ]:
# Chart 6a: Horizontal bar — SIP amount by state (top 12)
state_sip = (sip_txn.groupby('state')['amount_inr']
             .sum().sort_values(ascending=False).head(12).reset_index())
state_sip['amount_cr'] = state_sip['amount_inr'] / 1e7

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#1565C0' if s in ['Maharashtra','Delhi'] else '#90CAF9' for s in state_sip['state']]
bars = ax.barh(state_sip['state'][::-1], state_sip['amount_cr'][::-1], color=colors[::-1], edgecolor='white')
ax.set_title('Total SIP Investment by State (Top 12)', fontsize=13, fontweight='bold')
ax.set_xlabel('Total SIP Amount (₹ Crore)')
for bar, val in zip(bars, state_sip['amount_cr'][::-1]):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f'₹{val:.0f} Cr', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(CHARTS + '06a_sip_by_state_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# Chart 6b: T30 vs B30 pie
tier_counts = txn['city_tier'].value_counts().reset_index()
tier_counts.columns = ['tier','count']
fig2 = px.pie(tier_counts, values='count', names='tier',
              title='T30 vs B30 City Tier Split',
              color_discrete_map={'T30':'#1565C0','B30':'#90CAF9'},
              template=PLOTLY_TEMPLATE, hole=0.4)
fig2.update_traces(textposition='outside', textinfo='percent+label')
fig2.write_image(CHARTS + '06b_t30_b30_pie.png', width=600, height=450, scale=2)
fig2.show()

---
## Finding 7: Folio Count Doubled in 4 Years — from 13.26 Cr to 26.12 Cr
> The total investor folio count doubled from 13.26 Cr (Jan 2022) to 26.12 Cr (Dec 2025), with equity folios being the primary growth driver, crossing 18 Cr in late 2025.

In [ ]:
# Chart 7: Folio count growth with milestones
fig = go.Figure()

for col, name, color in [
    ('total_folios_crore',  'Total',   '#1565C0'),
    ('equity_folios_crore', 'Equity',  '#43A047'),
    ('debt_folios_crore',   'Debt',    '#FB8C00'),
    ('hybrid_folios_crore', 'Hybrid',  '#8E24AA'),
]:
    fig.add_trace(go.Scatter(
        x=folio['month'].dt.strftime('%Y-%m-%d').tolist(),
        y=folio[col].tolist(),
        mode='lines+markers', name=name,
        line=dict(color=color, width=2.5)
    ))

milestones = [
    ('2023-01-01', 14.81, '15 Cr'),
    ('2024-01-01', 17.78, '18 Cr'),
    ('2025-12-01', 26.12, '26 Cr ATH'),
]
for dt, val, label in milestones:
    fig.add_annotation(x=dt, y=val, text=f'<b>{label}</b>',
        showarrow=True, arrowhead=2, arrowcolor='#1565C0',
        font=dict(size=11, color='#1565C0'), ax=30, ay=-35)

fig.update_layout(
    title='Industry Folio Count Growth (Jan 2022 - Dec 2025)',
    xaxis_title='Month', yaxis_title='Folios (Crore)',
    template=PLOTLY_TEMPLATE, height=480
)
fig.write_image(CHARTS + '07_folio_count_growth.png', width=1400, height=480, scale=2)
fig.show()

---
## Finding 8: Large Cap Funds Show High Pairwise NAV Return Correlation
> The NAV return correlation matrix for 10 selected funds reveals that Large Cap funds (SBI, ICICI, HDFC, Axis, Kotak) are highly correlated (ρ > 0.85), confirming they track similar benchmark indices.

In [ ]:
# Chart 8: NAV return correlation matrix — 10 funds
selected = [119551, 119552, 120503, 120504, 100016, 125497, 118632, 119092, 120841, 148567]
labels   = [
    'SBI Bluechip Reg', 'SBI Bluechip Dir',
    'ICICI Bluechip Reg', 'ICICI Bluechip Dir',
    'HDFC Top100 Reg', 'HDFC Top100 Dir',
    'Nippon LargeCap', 'Axis Bluechip',
    'Kotak Bluechip', 'Mirae LargeCap'
]

nav_sel = (nav_trading[nav_trading['amfi_code'].isin(selected)]
           .pivot_table(index='date', columns='amfi_code', values='nav'))
returns = nav_sel.pct_change().dropna()
returns.columns = labels
corr = returns.corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=0.5, vmax=1.0, mask=mask,
            linewidths=0.5, ax=ax, square=True,
            cbar_kws={'shrink': 0.8, 'label': 'Pearson Correlation'})
ax.set_title('NAV Daily Return Correlation Matrix — 10 Selected Funds', fontsize=13, fontweight='bold')
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(CHARTS + '08_nav_return_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Finding 9: Banking & IT Sectors Dominate Equity Fund Portfolios
> Across all equity funds, Banking and IT together account for ~35–40% of aggregate portfolio weight, reflecting their combined dominance in the NIFTY 100 universe.

In [ ]:
# Chart 9: Sector allocation donut across all equity funds
eq_holdings = holdings[holdings['amfi_code'].isin(equity_codes)]
sector_wt   = eq_holdings.groupby('sector')['weight_pct'].sum().sort_values(ascending=False)
sector_pct  = (sector_wt / sector_wt.sum() * 100).reset_index()
sector_pct.columns = ['sector', 'pct']

# Group small sectors into 'Others'
threshold = 3.0
main = sector_pct[sector_pct['pct'] >= threshold]
others_val = sector_pct[sector_pct['pct'] < threshold]['pct'].sum()
if others_val > 0:
    main = pd.concat([main, pd.DataFrame({'sector':['Others'], 'pct':[others_val]})], ignore_index=True)

fig = px.pie(main, values='pct', names='sector',
             title='Aggregate Sector Allocation — All Equity Funds',
             template=PLOTLY_TEMPLATE, hole=0.45,
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_traces(textposition='outside', textinfo='percent+label')
fig.update_layout(height=520, showlegend=True)
fig.write_image(CHARTS + '09_sector_allocation_donut.png', width=900, height=520, scale=2)
fig.show()

---
## Finding 10: Small Cap Funds Deliver Highest 5-Year Returns but with Highest Drawdowns
> Small Cap funds average ~21–24% 5-year CAGR vs 11–15% for Large Cap, but their max drawdowns exceed –30% — highlighting the classic risk-return tradeoff in the Indian MF universe.

In [ ]:
# Chart 10: Return vs Drawdown scatter by sub-category
perf_eq = perf[perf['amfi_code'].isin(equity_codes)].merge(
    fund[['amfi_code','sub_category']], on='amfi_code', how='left')

fig = px.scatter(
    perf_eq, x='max_drawdown_pct', y='return_5yr_pct',
    color='sub_category', size='aum_crore',
    hover_data=['scheme_name','expense_ratio_pct'],
    title='5-Year Return vs Max Drawdown by Sub-Category',
    labels={'max_drawdown_pct':'Max Drawdown (%)', 'return_5yr_pct':'5-Year Return (%)'},
    template=PLOTLY_TEMPLATE, height=500
)
fig.update_traces(marker=dict(opacity=0.8, line=dict(width=1, color='white')))
fig.write_image(CHARTS + '10_return_vs_drawdown.png', width=1100, height=500, scale=2)
fig.show()

---
## Bonus Charts (15+ total)

In [ ]:
# Chart 11: Expense ratio distribution by sub-category
fig, ax = plt.subplots(figsize=(11, 5))
perf_m = perf.merge(fund[['amfi_code','sub_category']], on='amfi_code', how='left')
order = perf_m.groupby('sub_category')['expense_ratio_pct'].median().sort_values().index
sns.boxplot(data=perf_m, x='sub_category', y='expense_ratio_pct',
            order=order, palette='coolwarm', showfliers=True, ax=ax)
ax.axhline(1.0, color='orange', linestyle='--', label='1% threshold')
ax.set_title('Expense Ratio Distribution by Sub-Category', fontsize=13, fontweight='bold')
ax.set_xlabel('Sub-Category')
ax.set_ylabel('Expense Ratio (%)')
plt.xticks(rotation=30, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig(CHARTS + '11_expense_ratio_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 12: Top 10 funds by AUM — horizontal bar
top10_aum = perf.merge(fund[['amfi_code','scheme_name','sub_category']], on='amfi_code')\
                .nlargest(10, 'aum_crore')[['scheme_name','aum_crore','sub_category']]
top10_aum['aum_cr_k'] = top10_aum['aum_crore'] / 1000
top10_aum['short'] = top10_aum['scheme_name'].str[:35]

fig, ax = plt.subplots(figsize=(11, 6))
colors = sns.color_palette('Blues_d', len(top10_aum))
ax.barh(top10_aum['short'][::-1], top10_aum['aum_cr_k'][::-1], color=colors, edgecolor='white')
for i, (val, name) in enumerate(zip(top10_aum['aum_cr_k'][::-1], top10_aum['short'][::-1])):
    ax.text(val+100, i, f'₹{val:.0f}K Cr', va='center', fontsize=9)
ax.set_title('Top 10 Funds by AUM (₹ Thousand Crore)', fontsize=13, fontweight='bold')
ax.set_xlabel('AUM (₹ Thousand Crore)')
plt.tight_layout()
plt.savefig(CHARTS + '12_top10_aum.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 13: SIP growth YoY bar chart
sip['year'] = sip['month'].dt.year
sip_annual = sip.groupby('year')['sip_inflow_crore'].sum().reset_index()

fig = px.bar(sip_annual, x='year', y='sip_inflow_crore',
             title='Annual Total SIP Inflows (₹ Crore)',
             text='sip_inflow_crore',
             color='sip_inflow_crore', color_continuous_scale='Blues',
             template=PLOTLY_TEMPLATE)
fig.update_traces(texttemplate='₹%{text:,.0f}', textposition='outside')
fig.update_layout(height=430, coloraxis_showscale=False)
fig.write_image(CHARTS + '13_annual_sip_bar.png', width=900, height=430, scale=2)
fig.show()

In [ ]:
# Chart 14: Sharpe ratio comparison — top 15 equity funds
perf_eq14 = perf[perf['amfi_code'].isin(equity_codes)].merge(
    fund[['amfi_code','sub_category']], on='amfi_code', how='left')
top15_sharpe = perf_eq14.nlargest(15, 'sharpe_ratio')[['scheme_name','sharpe_ratio','sub_category']]
top15_sharpe['short'] = top15_sharpe['scheme_name'].str[:35]

fig, ax = plt.subplots(figsize=(11, 6))
pal = sns.color_palette('RdYlGn', len(top15_sharpe))
ax.barh(top15_sharpe['short'][::-1], top15_sharpe['sharpe_ratio'][::-1], color=pal, edgecolor='white')
ax.axvline(1.0, color='red', linestyle='--', label='Sharpe = 1.0')
ax.set_title('Top 15 Equity Funds by Sharpe Ratio', fontsize=13, fontweight='bold')
ax.set_xlabel('Sharpe Ratio')
plt.legend()
plt.tight_layout()
plt.savefig(CHARTS + '14_sharpe_ratio_top15.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 15: Transaction type breakdown — count and value
txn_type_count = txn['transaction_type'].value_counts().reset_index()
txn_type_count.columns = ['type','count']
txn_type_val = txn.groupby('transaction_type')['amount_inr'].sum().reset_index()
txn_type_val.columns = ['type','total_inr']
txn_type_val['total_cr'] = txn_type_val['total_inr'] / 1e7

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'pie'},{'type':'bar'}]],
                    subplot_titles=['Transaction Count Split', 'Total Value by Type (₹ Crore)'])
fig.add_trace(go.Pie(labels=txn_type_count['type'], values=txn_type_count['count'],
                     hole=0.4, marker_colors=['#1565C0','#43A047','#E53935']), row=1, col=1)
fig.add_trace(go.Bar(x=txn_type_val['type'], y=txn_type_val['total_cr'],
                     marker_color=['#1565C0','#43A047','#E53935'],
                     text=txn_type_val['total_cr'].round(0),
                     texttemplate='₹%{text:,.0f}', textposition='outside'), row=1, col=2)
fig.update_layout(title='Transaction Type Analysis', template=PLOTLY_TEMPLATE,
                  height=440, showlegend=False)
fig.write_image(CHARTS + '15_transaction_type_analysis.png', width=1200, height=440, scale=2)
fig.show()

---
## EDA Summary — 10 Key Findings

| # | Finding | Chart |
|---|---------|-------|
| 1 | All 40 funds rallied strongly through 2023 (bull run), followed by a broad market correction in Q3–Q4 2024 driven by FII outflows. | Chart 1 — NAV Trend |
| 2 | SBI Mutual Fund dominates industry AUM at ₹12.5L Cr (Mar 2025), nearly double its ₹6.05L Cr base in Mar 2022. | Chart 2 — AUM Bar |
| 3 | Monthly SIP inflows hit an all-time high of ₹31,002 Cr in Dec 2025, up 2.7× from ₹11,517 Cr in Jan 2022. | Chart 3 — SIP Timeseries |
| 4 | Liquid funds dominate monthly category inflows (₹33K–₹42K Cr), while Sectoral/Thematic surged in 2024–2025. | Chart 4 — Category Heatmap |
| 5 | The 36–45 age group has the highest SIP participation and median ticket size, representing wealth-accumulation phase investors. | Chart 5 — Demographics |
| 6 | Maharashtra and Delhi lead SIP investment value; T30 cities contribute ~75% of all transactions, confirming urban-led adoption. | Chart 6 — Geographic |
| 7 | Total investor folios doubled from 13.26 Cr (Jan 2022) to 26.12 Cr (Dec 2025), with equity folios crossing 18 Cr. | Chart 7 — Folio Growth |
| 8 | Large Cap fund NAV returns are highly correlated (ρ > 0.85), confirming they move in lockstep with NIFTY 100. | Chart 8 — Correlation Matrix |
| 9 | Banking and IT sectors together account for ~35–40% of aggregate equity fund portfolio weight. | Chart 9 — Sector Donut |
| 10 | Small Cap funds deliver 21–24% 5-year CAGR but sustain –30%+ max drawdowns, vs 11–15% returns and –17% to –25% drawdowns for Large Cap funds. | Chart 10 — Return vs Drawdown |


In [ ]:
import glob
charts = sorted(glob.glob(CHARTS + '*.png'))
print(f'Total PNG charts exported: {len(charts)}')
for c in charts:
    print(' ', c)